# Force–Deflection, Thermal Noise & Nanoscale Physics Simulator
## Interactive Companion to Chapter 1 — *Scanning Probe Microscopy*

This notebook accompanies **Chapter 1** of the textbook *Scanning Probe Microscopy*.

After completing this notebook you will be able to:

- Convert cantilever deflection into force using Hooke’s law and explore force regimes interactively.
- Understand Abbe’s diffraction limit and why SPM bypasses it.
- Visualise the exponential sensitivity of STM tunneling current.
- Estimate thermal RMS force and explore the sensitivity–noise trade-off.
- Use the Hertz contact model to estimate cell indentation forces.
- Perform order-of-magnitude sanity checks with an interactive dashboard.

**Central idea — At the nanoscale, measurement requires internal calibration.**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ---------- Physical constants ----------
k_B   = 1.380649e-23   # Boltzmann constant  (J/K)
T     = 298             # Room temperature     (K)
h_bar = 1.054571817e-34 # Reduced Planck const (J·s)
m_e   = 9.10938e-31     # Electron mass        (kg)
c     = 2.998e8         # Speed of light       (m/s)
h     = 6.626e-34       # Planck constant      (J·s)

print("Physical constants loaded ✓")

In [ ]:
# If widgets do not render, run this cell once (Colab).
!pip -q install ipywidgets

from ipywidgets import interact, FloatLogSlider, FloatSlider, IntSlider, Dropdown, HTML, fixed
import ipywidgets as widgets
from IPython.display import display, Markdown

print("Widgets ready ✓")

---
## 1. Hooke’s Law in AFM — Interactive Force Calculator

The cantilever behaves as a linear spring:

$$F = k \, \delta$$

where $F$ = force (N), $k$ = spring constant (N/m), $\delta$ = deflection (m).

⚠️ **Units matter.** AFM operates in nanometers, but equations require meters.

### Order-of-Magnitude Reference (Table 1.3 in textbook)

| Quantity | Typical Magnitude |
|---|---|
| Thermal RMS force | 5–30 pN |
| Single receptor–ligand bond | 10–200 pN |
| Soft-cell indentation force | 0.5–5 nN |
| Hard polymer indentation | 10–100 nN |

Use the sliders below to explore how spring constant and deflection determine the applied force, and see instantly which **force regime** your measurement falls into.


In [ ]:
def interactive_hooke(k_Nm=0.1, deflection_nm=15.0):
    delta_m = deflection_nm * 1e-9
    F = k_Nm * delta_m
    F_thermal = np.sqrt(k_B * T * k_Nm)
    SNR = F / F_thermal if F_thermal > 0 else np.inf

    # Force regime classification
    regimes = [
        (1e-12,  "Sub-pN (below thermal noise floor)"),
        (30e-12, "pN regime — molecular interactions"),
        (5e-9,   "nN regime — soft cell indentation"),
        (100e-9, "nN regime — hard polymer indentation"),
        (np.inf, "µN regime — likely unphysical for soft bio"),
    ]
    regime_label = ""
    for threshold, label in regimes:
        if F < threshold:
            regime_label = label
            break

    print(f"  Spring constant  k = {k_Nm:.4g} N/m")
    print(f"  Deflection       δ = {deflection_nm:.1f} nm")
    print(f"  ─────────────────────────────────")
    print(f"  Applied force    F = {F:.3e} N", end="  →  ")
    if F < 1e-9:
        print(f"{F*1e12:.2f} pN")
    elif F < 1e-6:
        print(f"{F*1e9:.3f} nN")
    else:
        print(f"{F*1e6:.3f} µN")
    print(f"  Thermal RMS force  = {F_thermal*1e12:.2f} pN")
    print(f"  Signal / Noise     = {SNR:.2f}")
    print()

    if SNR < 1:
        print("  ⚠ Signal BELOW thermal noise — measurement unreliable.")
    elif SNR < 5:
        print("  ⚠ Weak signal-to-noise — use caution.")
    else:
        print("  ✓ Strong signal-to-noise.")
    print(f"  Force regime: {regime_label}")

    # Bar chart visualisation
    fig, ax = plt.subplots(figsize=(8, 3))
    regime_bounds = [5e-12, 30e-12, 200e-12, 5e-9, 100e-9]
    regime_labels = ["Thermal\nnoise", "Molecular\nbonds", "Soft-cell\nindentation", "Hard\npolymer"]
    colors = ["#d9534f", "#f0ad4e", "#5cb85c", "#5bc0de"]
    log_bounds = np.log10(regime_bounds)

    for i in range(len(regime_labels)):
        ax.barh(0, log_bounds[i+1]-log_bounds[i], left=log_bounds[i],
                height=0.5, color=colors[i], alpha=0.6, edgecolor='k')
        ax.text((log_bounds[i]+log_bounds[i+1])/2, 0, regime_labels[i],
                ha='center', va='center', fontsize=8, fontweight='bold')

    log_F = np.log10(max(F, 1e-15))
    ax.axvline(log_F, color='red', lw=2, zorder=5)
    ax.text(log_F, 0.35, f"F = {F:.2e} N", ha='center', color='red', fontsize=9, fontweight='bold')
    ax.set_xlim(np.log10(1e-12), np.log10(1e-6))
    ax.set_xlabel("log₁₀(Force / N)")
    ax.set_yticks([])
    ax.set_title("Where does your force sit?")
    plt.tight_layout()
    plt.show()

_ = interact(
    interactive_hooke,
    k_Nm=FloatLogSlider(value=0.1, base=10, min=-2, max=2, step=0.1,
                        description='k (N/m)', style={'description_width': 'initial'}),
    deflection_nm=FloatSlider(value=15, min=0.1, max=500, step=0.5,
                              description='Deflection (nm)', style={'description_width': 'initial'})
)

In [ ]:
# Static reference plot: Force vs Deflection for multiple cantilever stiffnesses
k_values = [0.01, 0.05, 0.1, 0.5, 1.0, 10.0]
deflections_nm = np.linspace(0, 100, 300)

fig, ax = plt.subplots(figsize=(7, 4))
for k in k_values:
    forces_nN = k * deflections_nm * 1e-9 * 1e9
    ax.plot(deflections_nm, forces_nN, label=f"k = {k} N/m")

# Shade force regimes
ax.axhspan(0, 0.03, color='red',    alpha=0.08, label='Thermal noise (< 30 pN)')
ax.axhspan(0.03, 0.2, color='orange', alpha=0.08, label='Molecular bonds')
ax.axhspan(0.5, 5,   color='green',  alpha=0.08, label='Soft cell indentation')

ax.set_xlabel("Deflection (nm)")
ax.set_ylabel("Force (nN)")
ax.set_title("Force vs Deflection — Multiple Cantilever Stiffnesses")
ax.legend(fontsize=7, loc='upper left')
ax.set_ylim(0, 10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 2. Abbe’s Diffraction Limit — Why SPM Bypasses Wavelength

Optical microscopy resolution is fundamentally limited by diffraction (Section 1.2):

$$d = \frac{\lambda}{2 \, \text{NA}}$$

where $\lambda$ is the wavelength and $\text{NA} = n \sin\theta$ is the numerical aperture.

**Key insight from Chapter 1:** SPM bypasses this limit entirely because it does not use waves — it measures *local physical interactions*.

Use the sliders below to see how wavelength and NA affect the minimum resolvable distance.


In [ ]:
def interactive_abbe(wavelength_nm=500, NA=1.4):
    d_nm = wavelength_nm / (2 * NA)

    print(f"  Wavelength   λ  = {wavelength_nm:.0f} nm")
    print(f"  Numerical AP NA = {NA:.2f}")
    print(f"  ─────────────────────────────────")
    print(f"  Resolution limit d = {d_nm:.1f} nm")
    print()

    if d_nm > 200:
        print("  → Conventional optical microscopy regime")
    elif d_nm > 50:
        print("  → Super-resolution regime (STED, SIM)")
    elif d_nm > 10:
        print("  → Near super-resolution limit")
    else:
        print("  → Below practical optical limit")

    fig, ax = plt.subplots(figsize=(8, 3.5))

    techniques = {
        'AFM lateral':      0.1,
        'AFM vertical':     0.01,
        'STORM / PALM':     15,
        'STED':             30,
        'SIM':              100,
        'Optical (best)':   180,
    }
    y_pos = np.arange(len(techniques))
    colors_tech = ['#2ecc71', '#27ae60', '#3498db', '#9b59b6', '#e67e22', '#e74c3c']

    ax.barh(y_pos, [np.log10(v) for v in techniques.values()],
            color=colors_tech, alpha=0.7, edgecolor='k', height=0.6)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(techniques.keys())
    ax.set_xlabel("log₁₀(Resolution / nm)")

    ax.axvline(np.log10(d_nm), color='red', lw=2.5, ls='--', zorder=5)
    ax.text(np.log10(d_nm)+0.05, len(techniques)-0.5,
            f"Your limit: {d_nm:.0f} nm", color='red', fontsize=10, fontweight='bold')

    ax.set_title("Resolution Comparison — Abbe Limit vs Microscopy Techniques")
    ax.set_xlim(-2, 3)
    plt.tight_layout()
    plt.show()

_ = interact(
    interactive_abbe,
    wavelength_nm=FloatSlider(value=500, min=200, max=800, step=10,
                              description='λ (nm)', style={'description_width': 'initial'}),
    NA=FloatSlider(value=1.4, min=0.1, max=1.5, step=0.05,
                   description='NA', style={'description_width': 'initial'})
)

---
## 3. STM Tunneling Current — Exponential Distance Sensitivity

In Scanning Tunneling Microscopy (Section 1.3), the tunneling current depends **exponentially** on the tip–sample separation:

$$I \propto e^{-2\kappa z}$$

where $\kappa = \sqrt{2 m \phi / \hbar^2}$ is the decay constant and $\phi$ is the effective barrier height (work function).

For typical metals ($\phi \approx$ 4–5 eV), $\kappa \approx 10^{10}$ m⁻¹.

**Key insight:** A change of just **0.1 nm** reduces the current to ~37% of its value. This extreme sensitivity enables atomic resolution — but only on conductive samples.


In [ ]:
def interactive_stm(phi_eV=4.5, z_max_nm=1.5):
    phi_J = phi_eV * 1.602e-19   # eV to J
    kappa = np.sqrt(2 * m_e * phi_J) / h_bar
    z_nm = np.linspace(0.1, z_max_nm, 500)
    z_m = z_nm * 1e-9
    I_norm = np.exp(-2 * kappa * z_m)
    I_norm = I_norm / I_norm[0]  # Normalise to I(z_min) = 1

    print(f"  Work function φ = {phi_eV:.1f} eV")
    print(f"  Decay constant κ = {kappa:.3e} m⁻¹")
    print(f"  ─────────────────────────────────")
    dz = 0.1e-9
    decay_01 = np.exp(-2 * kappa * dz)
    print(f"  Current ratio after +0.1 nm: I/I₀ = {decay_01:.3f}  ({decay_01*100:.1f}%)")
    dz2 = 0.2e-9
    decay_02 = np.exp(-2 * kappa * dz2)
    print(f"  Current ratio after +0.2 nm: I/I₀ = {decay_02:.3f}  ({decay_02*100:.1f}%)")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

    # Linear scale
    ax1.plot(z_nm, I_norm, 'b-', lw=2)
    ax1.set_xlabel("Tip–sample distance z (nm)")
    ax1.set_ylabel("Normalised tunneling current I/I₀")
    ax1.set_title(f"Linear scale  (φ = {phi_eV} eV)")
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 1.05)
    z_ref = z_nm[0]
    ax1.axvline(z_ref + 0.1, color='red', ls='--', alpha=0.6)
    ax1.annotate(f"+0.1 nm → {decay_01*100:.0f}%", xy=(z_ref+0.1, decay_01),
                 xytext=(z_ref+0.3, 0.6), fontsize=9,
                 arrowprops=dict(arrowstyle='->', color='red'), color='red')

    # Log scale
    ax2.semilogy(z_nm, I_norm, 'b-', lw=2)
    ax2.set_xlabel("Tip–sample distance z (nm)")
    ax2.set_ylabel("Normalised tunneling current I/I₀")
    ax2.set_title(f"Log scale  (φ = {phi_eV} eV)")
    ax2.grid(True, alpha=0.3, which='both')

    plt.tight_layout()
    plt.show()

_ = interact(
    interactive_stm,
    phi_eV=FloatSlider(value=4.5, min=1.0, max=8.0, step=0.1,
                       description='φ (eV)', style={'description_width': 'initial'}),
    z_max_nm=FloatSlider(value=1.5, min=0.5, max=3.0, step=0.1,
                         description='z_max (nm)', style={'description_width': 'initial'})
)

---
## 4. Thermal Noise and the Sensitivity–Noise Trade-Off

From the **equipartition theorem** (Section 1.12.3.1):

$$\delta_{\text{RMS}} = \sqrt{\frac{k_B T}{k}}$$

The thermal RMS **force** noise is:

$$F_{\text{RMS}} = k \cdot \delta_{\text{RMS}} = \sqrt{k \, k_B T}$$

**Key insight from Chapter 1:** Softer cantilevers are more sensitive to small forces, but they also have larger thermal fluctuations. There is no “perfect” cantilever — every measurement is a trade-off.


In [ ]:
def interactive_thermal(k_min_exp=-2.0, k_max_exp=2.0, temperature_K=298):
    k_range = np.logspace(k_min_exp, k_max_exp, 500)
    delta_rms = np.sqrt(k_B * temperature_K / k_range)       # m
    F_rms = np.sqrt(k_range * k_B * temperature_K)            # N

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # (a) Thermal deflection
    ax = axes[0]
    ax.loglog(k_range, delta_rms * 1e9, 'b-', lw=2)
    ax.set_xlabel("k (N/m)")
    ax.set_ylabel("δ_RMS (nm)")
    ax.set_title("Thermal Deflection")
    ax.grid(True, which='both', alpha=0.3)
    ax.axvspan(0.01, 0.1, color='green', alpha=0.1, label='Bio cantilevers')
    ax.axvspan(1, 40, color='blue', alpha=0.1, label='Stiff cantilevers')
    ax.legend(fontsize=7)

    # (b) Thermal force noise
    ax = axes[1]
    ax.loglog(k_range, F_rms * 1e12, 'r-', lw=2)
    ax.set_xlabel("k (N/m)")
    ax.set_ylabel("F_RMS (pN)")
    ax.set_title("Thermal Force Noise")
    ax.grid(True, which='both', alpha=0.3)
    ax.axhspan(5, 30, color='orange', alpha=0.1, label='Typical 5–30 pN')
    ax.legend(fontsize=7)

    # (c) SNR for a fixed 10 nm deflection
    ax = axes[2]
    F_signal = k_range * 10e-9  # 10 nm deflection
    SNR = F_signal / F_rms
    ax.loglog(k_range, SNR, 'k-', lw=2)
    ax.axhline(1, color='red', ls='--', alpha=0.5, label='SNR = 1')
    ax.axhline(5, color='orange', ls='--', alpha=0.5, label='SNR = 5')
    ax.set_xlabel("k (N/m)")
    ax.set_ylabel("Signal / Noise")
    ax.set_title("SNR for δ = 10 nm")
    ax.grid(True, which='both', alpha=0.3)
    ax.legend(fontsize=7)

    plt.suptitle(f"T = {temperature_K} K", fontsize=11)
    plt.tight_layout()
    plt.show()

    # Summary table
    for k_test in [0.01, 0.1, 1.0, 40.0]:
        if 10**k_min_exp <= k_test <= 10**k_max_exp:
            d = np.sqrt(k_B * temperature_K / k_test)
            f = np.sqrt(k_test * k_B * temperature_K)
            print(f"  k = {k_test:6.2f} N/m  →  δ_RMS = {d*1e9:.3f} nm,  F_RMS = {f*1e12:.1f} pN")

_ = interact(
    interactive_thermal,
    k_min_exp=FloatSlider(value=-2, min=-3, max=0, step=0.5,
                          description='log₁₀(k_min)', style={'description_width': 'initial'}),
    k_max_exp=FloatSlider(value=2, min=0, max=3, step=0.5,
                          description='log₁₀(k_max)', style={'description_width': 'initial'}),
    temperature_K=FloatSlider(value=298, min=4, max=400, step=1,
                              description='T (K)', style={'description_width': 'initial'})
)

---
## 5. Hertz Contact Model — Estimating Cell Indentation Forces

From Section 1.6 and the numerical exercises, the **Hertz model** for a spherical indenter gives:

$$F = \frac{4}{3} E^* \sqrt{R} \, \delta^{3/2}$$

where $E^*$ is the reduced modulus, $R$ is the tip radius, and $\delta$ is the indentation depth.

For a rigid tip on a soft sample: $E^* \approx E_{\text{sample}} / (1 - \nu^2)$.

**Typical values from Chapter 1:**
- Soft biological cells: $E \sim$ 0.5–10 kPa
- Tip radius: 5–20 µm (colloidal probe) or 10–50 nm (sharp tip)
- Indentation: 50–500 nm


In [ ]:
def interactive_hertz(E_kPa=5.0, R_um=5.0, delta_nm=200, nu=0.5):
    E_Pa = E_kPa * 1e3
    E_star = E_Pa / (1 - nu**2)
    R_m = R_um * 1e-6
    delta_m = delta_nm * 1e-9
    F = (4.0/3.0) * E_star * np.sqrt(R_m) * delta_m**1.5

    print(f"  Young's modulus   E = {E_kPa:.1f} kPa")
    print(f"  Poisson's ratio   ν = {nu:.2f}")
    print(f"  Reduced modulus  E* = {E_star:.1f} Pa")
    print(f"  Tip radius        R = {R_um:.1f} µm")
    print(f"  Indentation       δ = {delta_nm:.0f} nm")
    print(f"  ─────────────────────────────────")
    print(f"  Hertz force       F = {F:.3e} N", end="  →  ")
    if F < 1e-9:
        print(f"{F*1e12:.1f} pN")
    elif F < 1e-6:
        print(f"{F*1e9:.3f} nN")
    else:
        print(f"{F*1e6:.3f} µN")

    print()
    if 0.5e-9 <= F <= 5e-9:
        print("  ✓ Consistent with soft-cell indentation range (0.5–5 nN)")
    elif F < 0.5e-9:
        print("  ⚠ Below typical soft-cell range — very soft or small indentation")
    elif F <= 100e-9:
        print("  ⚠ Above soft-cell range — entering hard polymer territory")
    else:
        print("  ⚠ Well above typical AFM range — check parameters")

    # Plot: Force vs indentation for current E and R
    deltas = np.linspace(0, 500, 300) * 1e-9
    forces = (4.0/3.0) * E_star * np.sqrt(R_m) * deltas**1.5

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

    ax1.plot(deltas*1e9, forces*1e9, 'b-', lw=2)
    ax1.axhline(0.5, color='green', ls='--', alpha=0.5, label='0.5 nN (soft cell min)')
    ax1.axhline(5.0, color='green', ls='-',  alpha=0.5, label='5 nN (soft cell max)')
    ax1.axvline(delta_nm, color='red', ls='--', alpha=0.7)
    ax1.plot(delta_nm, F*1e9, 'ro', ms=10, zorder=5)
    ax1.set_xlabel("Indentation δ (nm)")
    ax1.set_ylabel("Force (nN)")
    ax1.set_title(f"Hertz: F vs δ  (E={E_kPa} kPa, R={R_um} µm)")
    ax1.legend(fontsize=7)
    ax1.grid(True, alpha=0.3)

    # Force vs E for current delta and R
    E_range_kPa = np.logspace(-1, 2, 300)
    E_range_Pa = E_range_kPa * 1e3
    E_star_range = E_range_Pa / (1 - nu**2)
    F_range = (4.0/3.0) * E_star_range * np.sqrt(R_m) * delta_m**1.5

    ax2.loglog(E_range_kPa, F_range*1e9, 'b-', lw=2)
    ax2.axhline(0.5, color='green', ls='--', alpha=0.5)
    ax2.axhline(5.0, color='green', ls='-',  alpha=0.5)
    ax2.axvline(E_kPa, color='red', ls='--', alpha=0.7)
    ax2.plot(E_kPa, F*1e9, 'ro', ms=10, zorder=5)
    ax2.set_xlabel("Young's modulus E (kPa)")
    ax2.set_ylabel("Force (nN)")
    ax2.set_title(f"Hertz: F vs E  (δ={delta_nm} nm, R={R_um} µm)")
    ax2.grid(True, which='both', alpha=0.3)

    plt.tight_layout()
    plt.show()

_ = interact(
    interactive_hertz,
    E_kPa=FloatLogSlider(value=5, base=10, min=-1, max=2, step=0.1,
                         description='E (kPa)', style={'description_width': 'initial'}),
    R_um=FloatSlider(value=5.0, min=0.01, max=25, step=0.5,
                     description='R (µm)', style={'description_width': 'initial'}),
    delta_nm=FloatSlider(value=200, min=10, max=1000, step=10,
                         description='δ (nm)', style={'description_width': 'initial'}),
    nu=FloatSlider(value=0.5, min=0.2, max=0.5, step=0.01,
                   description='ν (Poisson)', style={'description_width': 'initial'})
)

---
## 6. Sanity-Check Dashboard — Order-of-Magnitude Reasoning

From the exercises in Section 1.8: A student reports that indenting a live cell by 100 nm required **80 µN** of force. The order-of-magnitude table tells us this is physically implausible.

Use this interactive dashboard to enter **any** reported force result and check it against the physical reference table. This trains the **engineering judgment** emphasised throughout Chapter 1.

> *“At the nanoscale, precision without physical judgment is misleading.”*


In [ ]:
def sanity_dashboard(k_Nm=0.1, deflection_nm=100, reported_force_nN=1.5):
    F_computed = k_Nm * deflection_nm * 1e-9
    F_thermal  = np.sqrt(k_B * T * k_Nm)
    F_reported = reported_force_nN * 1e-9

    SNR_computed = F_computed / F_thermal if F_thermal > 0 else np.inf
    SNR_reported = F_reported / F_thermal if F_thermal > 0 else np.inf

    print("  ╔══════════════════════════════════════════════╗")
    print("  ║       SANITY-CHECK DASHBOARD                ║")
    print("  ╠══════════════════════════════════════════════╣")
    print(f"  ║  Cantilever k       = {k_Nm:.4g} N/m")
    print(f"  ║  Deflection         = {deflection_nm:.0f} nm")
    print(f"  ║  Computed force      = {F_computed:.3e} N  ({F_computed*1e9:.3f} nN)")
    print(f"  ║  Reported force      = {F_reported:.3e} N  ({reported_force_nN:.3f} nN)")
    print(f"  ║  Thermal noise floor = {F_thermal*1e12:.1f} pN")
    print(f"  ║  SNR (computed)      = {SNR_computed:.1f}")
    print(f"  ║  SNR (reported)      = {SNR_reported:.1f}")
    print("  ╚══════════════════════════════════════════════╝")
    print()

    ratio = F_reported / F_computed if F_computed > 0 else np.inf
    if 0.1 < ratio < 10:
        print(f"  ✓ Reported force is within 1 order of magnitude of computed value (ratio = {ratio:.2f})")
    else:
        print(f"  ⚠ MISMATCH: Reported / Computed = {ratio:.2e}")
        print("    Possible errors: unit conversion, nm↔m, deflection↔indentation confusion")

    # Visual
    fig, ax = plt.subplots(figsize=(9, 3.5))
    regimes = [
        ("Thermal\nnoise", 5e-12, 30e-12, '#d9534f'),
        ("Molecular\nbonds", 30e-12, 200e-12, '#f0ad4e'),
        ("Soft cell", 0.5e-9, 5e-9, '#5cb85c'),
        ("Hard\npolymer", 10e-9, 100e-9, '#5bc0de'),
    ]
    for label, lo, hi, col in regimes:
        ax.axvspan(np.log10(lo), np.log10(hi), color=col, alpha=0.25)
        ax.text((np.log10(lo)+np.log10(hi))/2, 0.85, label,
                ha='center', va='top', fontsize=8, transform=ax.get_xaxis_transform())

    ax.axvline(np.log10(max(F_computed, 1e-15)), color='blue', lw=2.5, label=f'Computed: {F_computed*1e9:.3f} nN')
    ax.axvline(np.log10(max(F_reported, 1e-15)), color='red', lw=2.5, ls='--', label=f'Reported: {reported_force_nN:.3f} nN')
    ax.axvline(np.log10(F_thermal), color='gray', lw=1.5, ls=':', label=f'Thermal: {F_thermal*1e12:.1f} pN')

    ax.set_xlim(-12.5, -5)
    ax.set_xlabel("log₁₀(Force / N)")
    ax.set_yticks([])
    ax.set_title("Force Comparison Against Physical Reference Regimes")
    ax.legend(fontsize=8, loc='upper right')
    plt.tight_layout()
    plt.show()

_ = interact(
    sanity_dashboard,
    k_Nm=FloatLogSlider(value=0.1, base=10, min=-2, max=2, step=0.1,
                        description='k (N/m)', style={'description_width': 'initial'}),
    deflection_nm=FloatSlider(value=100, min=1, max=500, step=1,
                              description='Deflection (nm)', style={'description_width': 'initial'}),
    reported_force_nN=FloatLogSlider(value=1.5, base=10, min=-3, max=5, step=0.1,
                                     description='Reported F (nN)', style={'description_width': 'initial'})
)

---
## 7. Engineering Insight — Resolution vs Reliability

From the Engineering Insight Box in Chapter 1:

> Every AFM experiment is a **multi-parameter optimisation** involving spatial resolution, force sensitivity, measurement bandwidth, sample perturbation, and model validity.

**Key trade-offs explored in this notebook:**

| Softer cantilever (↓ k) | Stiffer cantilever (↑ k) |
|---|---|
| ↑ Force sensitivity | ↓ Force sensitivity |
| ↑ Thermal deflection noise | ↓ Thermal deflection noise |
| Suitable for cells, molecules | Suitable for polymers, hard materials |
| Risk: SNR too low | Risk: sample damage |

**Take-away:** Higher spatial resolution does not automatically imply higher mechanical accuracy. Numerical precision alone does not guarantee physical accuracy.


---
## Summary

This notebook has covered the key quantitative concepts from Chapter 1:

1. **Hooke’s Law** — Force from cantilever deflection, with force regime classification.
2. **Abbe’s Diffraction Limit** — Why optical resolution is wavelength-limited and SPM bypasses it.
3. **STM Tunneling Current** — Exponential distance sensitivity enabling atomic resolution.
4. **Thermal Noise** — The fundamental sensitivity–noise trade-off in cantilever selection.
5. **Hertz Contact Model** — Estimating indentation forces on soft biological samples.
6. **Sanity-Check Dashboard** — Order-of-magnitude reasoning for validating experimental results.

### Core Message

> **At the nanoscale, measurement is interaction.**

Reliable AFM measurements require balancing sensitivity, stability, and physical noise limits. Engineering judgment remains essential — *precision without physical judgment is misleading.*
